# Day 11 | ILT 1: Orchestration in Practice — A Live Walkthrough of Jobs & Pipelines
### GlobalMart Data Engineering Bootcamp
---
**Duration:** 90 minutes &nbsp;|&nbsp; **Level:** Intermediate &nbsp;|&nbsp; **Tags:** orchestration, databricks-workflows, jobs-ui, cluster-config, parameters, notifications, retries

---
**Builds on:** Day 10 ILT 2 (*Introduction to Orchestration — Need, DAG Concepts and Workflow Design*) and Day 10 ILT 3 (*Databricks Workflows vs. Airflow — Deep Dive*). Day 10 answered **why** you orchestrate and **what** a DAG is. This session answers **how** — the actual screens, fields, and decisions inside Databricks Jobs & Pipelines.

**Format:** this is a **live, on-screen walkthrough** — the instructor builds one real Job in the Databricks UI, live, narrating every field. This notebook is the script for that walkthrough plus the reference material students keep afterward. Nothing in this notebook creates, starts, or schedules anything itself — every Jobs/cluster setting below is shown as an inspectable Python dictionary, matching what you'll actually see in the UI, never a real API call.

## By the End of This Session You Will Be Able To

1. Navigate to Jobs & Pipelines and create a Job with a Notebook task from scratch
2. Choose and justify every compute setting for a job cluster — name, Photon, node type
3. Explain what a dependent library is, when a notebook needs one, and how to add it correctly
4. Use job parameters the *right* way — and explain, with a concrete bad example, the one thing that must never go in a parameter
5. Configure notifications, retries, and metric thresholds with real GlobalMart-shaped reasoning behind every setting

## Why This Session Exists Before the Hands-On Task

You're about to be handed a task: take a real GlobalMart notebook and orchestrate it. Before that, you need to have actually *seen* every screen you'll be clicking through — not read about them. This walkthrough goes in the exact order you'll repeat yourself: **Jobs & Pipelines → Jobs → Create Job → Notebook task → compute → libraries → parameters → notifications → retries → metric thresholds.**

---
## Section 1 — Getting There: Jobs & Pipelines

**INSTRUCTOR — live on screen:**

1. Left sidebar → **Jobs & Pipelines**
2. You'll see two tabs: **Jobs** and **Pipelines** (Lakeflow Declarative Pipelines — the DLT-style engine, a different tool for a different job, not what we're using today)
3. Stay on **Jobs** → click **Create Job**

| | |
|---|---|
| **Jobs** | You author the steps yourself: which notebook runs, in what order, on what compute. This is what we're doing today. |
| **Pipelines** | You declare the *end state* of your tables (`CREATE STREAMING TABLE`, `CREATE MATERIALIZED VIEW`) and the engine figures out the execution plan and incremental refresh for you. Different mental model — not covered today. |

You land on a blank job with one default task. Everything below is filling in that one task, then the settings around it.

In [ ]:
# What you're about to build, as data -- this is the SAME shape the Jobs UI
# saves under the hood when you click through the screens below. Printed here
# so you can see the whole picture before diving into each field one at a time.
# This is a plain Python dict, inspected and printed -- never submitted to a
# real Jobs API. Per this repo's safety rules, no notebook creates real jobs.

orchestration_job_plan = {
    "job_name": "globalmart_orchestration_demo",
    "tasks": [
        {
            "task_key": "orchestration",              # <- Task name field
            "notebook_task": {
                "notebook_path": "/Workspace/Users/<you>/Databricks/Day4/Day4_3_HOL1_Build_Bronze_Layer_Mounting",
                "source": "WORKSPACE",
            },
            "new_cluster": {                            # <- "Add new cluster" compute
                "cluster_name": "orchestration-demo-cluster",
                "spark_version": "15.4.x-scala2.12",
                "node_type_id": "Standard_DS3_v2",       # 14 GB RAM, 4 cores
                "num_workers": 1,
                "runtime_engine": "STANDARD",            # STANDARD = Photon OFF; PHOTON = Photon ON
            },
            "libraries": [],                             # dependent libraries -- empty for this demo
            "parameters": {},                             # job parameters -- filled in Section 4
        }
    ],
}

print(json.dumps(orchestration_job_plan, indent=2))

In [ ]:
import json  # used by the cell above; kept explicit rather than assumed

---
## Section 2 — The Task: Name, Type, Source, Path

**INSTRUCTOR — live on screen, filling in the first task:**

| Field | What you enter | Why |
|---|---|---|
| **Task name** | `orchestration` | Every task needs a unique name within the job — this becomes the `task_key` other tasks reference in `depends_on` (Day 10 ILT 3 covered this). Name it after *what it does*, not "task1" — in a real pipeline you'd see names like `ingest_bronze_customers`, `build_silver`, `build_gold`. |
| **Type** | `Notebook` | Databricks Jobs supports many task types — Notebook, Python script, JAR, SQL, dbt, Pipeline, another Job (for nesting), a plain shell command. **Notebook** is what almost every task in this course would be, since that's how we've built everything. |
| **Source** | `Workspace` | Where the notebook lives. `Workspace` means "browse the same workspace file tree you already use." The other option, `Git provider`, points the task at a specific commit/branch in a linked repo instead — the production-grade choice, since it decouples what a job *runs* from whoever's editing the notebook live in the Workspace right now. We use `Workspace` today because we're demoing, not shipping. |
| **Path** | e.g. `/Workspace/Users/<you>/Databricks/Day4/Day4_3_HOL1_Build_Bronze_Layer_Mounting` | The exact notebook this task runs. Browse to it with the file picker — don't type the path by hand, it's easy to get subtly wrong. |

**Common mistake to call out live:** picking `Source: Git provider` without a repo actually linked yet — the UI will block you until you connect one under **Workspace settings → Git integration**. Worth showing what that error looks like once, so it's not a surprise later.

---
## Section 3 — Compute: "Add New Cluster"

**INSTRUCTOR — live on screen:**

Under the task, the **Compute** dropdown offers three choices: an existing all-purpose cluster, a Databricks SQL warehouse (for SQL-only tasks), or **Add new cluster**. Click **Add new cluster**.

### Why a *new* cluster, not your existing all-purpose one?

This is the single most important cost decision on this whole screen, and it's why Day 11 ILT 2 (Cost Awareness) exists as its own session.

| | All-purpose cluster | Job cluster (what "Add new cluster" creates) |
|---|---|---|
| **Lifetime** | Stays running until *you* stop it or it idles out | Starts when the job starts, **terminates automatically** when the job ends |
| **Who else uses it** | Anyone attached to it, any time | Nobody — exclusive to this job run |
| **DBU rate** | Higher (interactive workload pricing) | Lower (jobs-compute pricing) |
| **Right for** | Development, exploring data live | Scheduled, unattended production runs |

A job you schedule to run every night should **never** point at your personal all-purpose cluster — it's more expensive per DBU, and if you're mid-edit on it when the schedule fires, you've just contended for compute with your own scheduled pipeline.

### The fields, one at a time

**Cluster name** — `orchestration-demo-cluster` here. In a real pipeline, name it after the *job*, not generically ("cluster1"), so anyone looking at the Compute tab six months from now can tell which job it belongs to.

**Databricks Runtime Version** — pick a current LTS version (e.g. `15.4.x-scala2.12`). Matters because it pins the Spark version, and therefore which Delta/Photon features are available.

**Use Photon Acceleration — deselect it for this demo.**

Photon is Databricks' native, vectorized query engine — a C++ rewrite of Spark's execution engine that dramatically speeds up SQL-shaped and DataFrame-shaped workloads (joins, aggregations, scans) at a **higher DBU rate per hour**.

- **Deselect it when:** the workload is light (like this demo), dominated by non-Photon-eligible work (heavy UDFs, ML training, simple file-copy style tasks), or you're cost-optimizing a job that isn't on the critical path.
- **Select it when:** the job is genuinely SQL/DataFrame-heavy at real data volume — exactly the shape of GlobalMart's own Bronze → Silver → Gold builds. Day 7 ILT 2 (*Performance Modelling*) already covered Photon's actual speed impact on `fact_sales`-shaped joins; the decision to enable it there is a cost-vs-speed tradeoff, not a default.

**Worker type / Node type — `Standard_DS3_v2` (14 GB RAM, 4 cores).**

This is an Azure VM size. Reading the name: `DS3_v2` is a general-purpose VM family; 14 GB RAM and 4 cores is its actual spec. For a demo job touching a handful of small tables, this is comfortably enough — you're not shuffling GBs of data. Sizing rule of thumb: match node memory to your **largest single-task working set**, not your whole warehouse. A job scanning all 8 Bronze tables in one wide join needs a bigger node than one reading a single small dimension.

**Min/Max workers** — 1 worker (no autoscaling) is fine for a demo-sized job. Autoscaling (a min/max range) matters once data volume varies run to run — e.g. a monthly batch job that's 10x bigger on month-end.

In [ ]:
# The compute block from the plan above, isolated so you can see exactly
# which UI field maps to which key.
compute_config = {
    "cluster_name": "orchestration-demo-cluster",
    "spark_version": "15.4.x-scala2.12",       # Databricks Runtime Version
    "node_type_id": "Standard_DS3_v2",          # Worker type: 14 GB RAM, 4 cores
    "num_workers": 1,
    "runtime_engine": "STANDARD",               # STANDARD = Photon deselected
    "autotermination_minutes": 20,               # only matters for all-purpose; job clusters
                                                   # already terminate when the job ends
}
print(json.dumps(compute_config, indent=2))

---
## Section 4 — Dependent Libraries

**INSTRUCTOR — live on screen:** on the task's **Libraries** tab, click **Add** — a library can come from PyPI, Maven, a `.whl`/`.egg` file uploaded to a Volume, or a Workspace file.

### What is a "dependent library," actually?

The Databricks Runtime ships with a large set of pre-installed packages — `pyspark`, `pandas`, `numpy`, `delta-spark`, and more, already on every cluster you attach to. A **dependent library** is anything your notebook imports that is *not* on that pre-installed list. If a task's cluster doesn't have it, the notebook fails at the `import` line with `ModuleNotFoundError` — every time, on every run, since a job cluster starts fresh with nothing carried over from your last interactive session.

### Real GlobalMart examples of when you'd actually need one

| Library | Why a GlobalMart notebook might need it | Source |
|---|---|---|
| `openpyxl` | Reading `.xlsx` files directly (e.g. the Data Dictionary/Star Schema workbooks from the supply chain assessment) — not pre-installed, `pandas.read_excel` fails without it | PyPI |
| A Postgres JDBC driver | Only needed if you're hand-rolling a JDBC connection to Supabase yourself (like the teaching demo in Day 2 ILT 1) rather than going through the managed Lakeflow Connect pipeline | Maven |
| `azure-storage-blob` | Direct Python-SDK access to ADLS Blob Storage, bypassing Spark's own `abfss://` reader — rare, but shows up in custom tooling/scripts | PyPI |

Notice: nothing in the **actual, real Bronze/Silver/Gold pipeline notebooks** built across this course needs an extra library — `pyspark.sql.functions`, `delta.tables`, and the pre-installed runtime cover all of it. That's not an accident; it's a sign the pipeline is built on the platform's native tools rather than bolted-on packages. Reach for a dependent library when you have a *specific, named reason* (a file format Spark can't read natively, a driver for a system Spark doesn't talk to directly) — not by default.

### Cluster-scoped vs. job-scoped

A library added on the **task's** Libraries tab installs only for that task's job cluster, only for that run. If ten tasks in one job all need `openpyxl`, you'd add it to each one, or better, use a cluster policy / init script if it's genuinely a standing requirement across your whole team's jobs. Pin a version (`openpyxl==3.1.2`, not just `openpyxl`) — an unpinned library can silently change behavior when PyPI ships a new release between two runs of the same job, weeks apart, with no code change on your end.

In [ ]:
# Illustrative only -- shows what a pinned PyPI library entry looks like in
# the Jobs config, matching what you'd see after clicking "Add" on the
# Libraries tab and choosing PyPI.
library_entry_example = {
    "pypi": {
        "package": "openpyxl==3.1.2"   # <- always pin the version
    }
}
print(json.dumps(library_entry_example, indent=2))

---
## Section 5 — Parameters: the One Rule That Matters Most Today

**INSTRUCTOR — live on screen:** on the task's **Parameters** tab, add a key/value pair. Inside the notebook, that value is read with `dbutils.widgets.get("key_name")`.

### Why parameters exist

The whole point of a parameter is to make **one notebook** work correctly in more than one situation, without editing its code. Day 11 ILT 2 (Cost Awareness & Environment Strategy) teaches a dev/test/prod environment split using separate Unity Catalogs — parameters are *how* the same notebook targets a different catalog in each environment.

### Good example — `catalog_name`

```
Key:   catalog_name
Value: harsh_kumar01_npmentorskool_onmicrosoft_com
```

Inside the notebook:
```python
CATALOG = dbutils.widgets.get("catalog_name")
df = spark.table(f"{CATALOG}.silver.orders")
```

Run this exact same job against a `_dev` catalog by changing one parameter value — no code edit, no redeploy. This is the entire reason parameters exist, and a catalog name is a perfect example: it identifies *where*, it isn't secret, and it's genuinely different across dev/test/prod.

### Bad example — never do this: `storage_account_key`

```
Key:   storage_account_key
Value: <a real 88-character Azure storage key>          ❌ NEVER
```

**Why this is a real, serious mistake, not just a style preference:**

- Job parameters are stored in **plaintext** in the job configuration and every run's history. Anyone with permission to *view* the job — not even run it — can read the value straight off the Job Details or Run Details screen.
- It shows up in **audit logs** and in the job's JSON definition if anyone exports it.
- There's no way to rotate the key without editing every job that references it by value.
- This is the exact class of mistake this project's own real pipeline notebooks hit once — a hardcoded Azure storage key was found sitting in a committed notebook earlier this cohort's build, caught only because GitHub's push protection blocked it. A parameter field is just as public as a code comment; it is not a safe place to put a secret.

### The correct way: a Secret Scope

```python
storage_key = dbutils.secrets.get(scope="globalmart-storage", key="account-key")
```

The **scope name** and **key name** (`globalmart-storage`, `account-key`) are safe to put in a parameter — they're just identifiers, not the secret itself. The actual value lives in a Secret Scope (backed by Azure Key Vault or Databricks-managed secrets), and `dbutils.secrets.get()` automatically **redacts it from all output** — if you `print(storage_key)`, Databricks prints `[REDACTED]`, not the real value, in every notebook cell output, log, and job run detail.

| | Job parameter | Secret Scope |
|---|---|---|
| **Right for** | Anything non-sensitive that varies by run — a catalog, a date, a table name, a row-count threshold | Anything that grants access — a key, a password, a token, a connection string |
| **Visible to** | Anyone who can view the job | Only code with an explicit `dbutils.secrets.get()` call, and even then the *value* is redacted from output |
| **GlobalMart precedent** | `CATALOG`, `SILVER_TABLE`, `LAST_PROCESSED_VERSION` in every incremental notebook this course has built | The storage account key setup in Day 4's mounting notebook explicitly instructs: paste your real key to test, then replace it back with a placeholder before saving — never leave a real key in a notebook or a parameter anyone else can open |

In [ ]:
# Two parameter blocks, side by side -- one safe, one deliberately shown as
# what NOT to do. The unsafe one uses an obviously-fake placeholder string,
# never a real credential, even for illustration.

safe_parameters = {
    "catalog_name": "harsh_kumar01_npmentorskool_onmicrosoft_com",
    "environment": "dev",
}

unsafe_parameters_DO_NOT_DO_THIS = {
    "storage_account_key": "<NEVER PUT A REAL KEY HERE — this is the mistake, not the fix>",
}

correct_secret_reference = {
    "secret_scope": "globalmart-storage",   # <- the SCOPE NAME is safe to pass as a parameter
    "secret_key": "account-key",             # <- the KEY NAME is safe to pass as a parameter
    # the actual secret VALUE never appears in a parameter, a notebook cell, or a print()
}

print("Safe parameters:", json.dumps(safe_parameters, indent=2))
print("\nCorrect way to reference a secret (names only, never the value):")
print(json.dumps(correct_secret_reference, indent=2))

---
## Section 6 — Notifications, In Detail

**INSTRUCTOR — live on screen:** notifications live in two places — **Job-level** (Job details → Edit notifications, applies to the whole job) and **Task-level** (per task, for finer control). Click **Add notification** and walk through the event types.

### The event types

| Event | Fires when | GlobalMart example |
|---|---|---|
| **On start** | The run begins | Rarely used — mostly noise unless you specifically need a "run started" audit trail |
| **On success** | Every task in the run completes without error | Turn this **off** for routine nightly Bronze/Silver/Gold runs (nobody needs an email every night something worked); turn it **on** for something rare and high-stakes, like a one-time backfill job |
| **On failure** | Any task fails | **Always on**, for every production job. This is the one that actually matters. |
| **On duration warning** | The run exceeds a duration threshold you set, *even if it eventually succeeds* | Set this on the Gold `fact_sales` refresh: if it normally finishes in ~10 minutes and one night takes 40, that's worth knowing *before* someone notices the dashboard looks stale |
| **On streaming backlog exceeded** | A streaming task's backlog (unprocessed data behind the latest offset) crosses a threshold | Relevant for the Bronze Autoloader/streaming-table tasks specifically — a growing backlog means ingestion is falling behind the source |

### Destinations

- **Email** — simplest, works everywhere, easy to spam yourself into ignoring
- **Notification destination** (Slack, Microsoft Teams, PagerDuty, generic webhook) — configured once centrally by a workspace admin under **Workspace settings → Notification destinations**, then any job can point at it. This is the real-team pattern: failures land in a shared `#data-eng-alerts` channel, not one person's inbox.

### A concrete GlobalMart notification plan

```
Bronze ingestion job     → On failure only               → #data-eng-alerts
Silver transform job     → On failure + duration warning → #data-eng-alerts
Gold fact_sales refresh  → On failure + duration warning → #data-eng-alerts + email to pipeline owner
One-time backfill job    → On start + on success + on failure → email to whoever's running it
```

Notice the pattern: routine, frequent jobs get quiet unless something's actually wrong (failure/duration-only). Rare, high-stakes, one-off jobs get chattier notifications because there's a human specifically watching that one run.

In [ ]:
notification_settings_example = {
    "on_failure": ["data-eng-alerts-webhook"],
    "on_duration_warning_threshold_exceeded": ["data-eng-alerts-webhook"],
    "on_success": [],   # deliberately empty -- routine job, no need to notify on every success
    "health": {
        "rules": [
            {"metric": "RUN_DURATION_SECONDS", "op": "GREATER_THAN", "value": 600}
            # fires the duration-warning notification above if this run takes > 10 min
        ]
    },
}
print(json.dumps(notification_settings_example, indent=2))

---
## Section 7 — Retries, In Detail

**INSTRUCTOR — live on screen:** each task has its own **Retry policy** section — number of retries, and the minimum interval between attempts.

### The settings

| Setting | What it controls |
|---|---|
| **Retries** | How many times a failed task automatically re-runs before the job is marked failed for good |
| **Minimum retry interval** | How long to wait before the next attempt — gives a transient problem time to clear |
| **Retry on timeout** | Whether a task that ran too long (rather than erroring) also counts as a failure eligible for retry |

### When a retry genuinely helps

Retries exist for **transient** failures — problems that are true right now and false a minute later, with no code change:

- A brief network blip reaching Supabase/Postgres
- The cluster was still spinning up when the first task tried to attach
- A momentary rate limit from a source API (Day 3's REST API ingestion, for example)

For these, 2–3 retries with a short interval (1–2 minutes) is standard, and the second attempt usually just works.

### When a retry actively makes things worse — the idempotency trap

**A retry is only safe if re-running the same task twice produces the same result as running it once.** This is exactly the property this course spent real effort fixing this cohort: the `orders_incremental_scd1_merge` notebook originally had a bug where its CDF read could hand the MERGE two rows for the same `order_id` (an `update_preimage` and an `update_postimage`), and separately, a Step 5b insert block was accidentally left duplicated three times in the Silver `customers` SCD2 notebook — running that top-to-bottom would triple-insert every changed customer row.

Now imagine either of those bugs on a job with **retries turned on**. The MERGE fails, the platform automatically retries it, and if the underlying bug were "sometimes inserts a row twice" rather than "always crashes," a retry wouldn't just fail loudly again — it could succeed on the second attempt while having *already* partially written duplicate data on the first. Retries don't cause bad idempotency; they make an existing idempotency bug happen silently, twice, instead of loudly, once.

**The rule to teach here:** before turning retries on for a task, ask "if this task runs successfully twice in a row on the same input, is that safe?" For a MERGE keyed on a real primary key with `whenMatchedUpdateAll()`/`whenNotMatchedInsertAll()` — yes, safe, matching rows just get overwritten with identical data. For a blind `INSERT`/`append` with no key — no, not safe, retries (or even just re-running the job by hand) will duplicate every row.

### A concrete GlobalMart retry plan

```
Bronze Autoloader task        → 2 retries, 2 min interval   (transient source/network issues)
Silver MERGE tasks            → 2 retries, 2 min interval   (safe -- MERGE is idempotent by key)
Gold aggregate overwrite task → 1 retry, 5 min interval     (overwrite is naturally idempotent)
Any task with a known, unfixed idempotency bug → 0 retries  (fix the bug first; a retry just hides it)
```

In [ ]:
retry_policy_example = {
    "min_retry_interval_millis": 120000,   # 2 minutes
    "max_retries": 2,
    "retry_on_timeout": True,
}
print(json.dumps(retry_policy_example, indent=2))

print()
print("Idempotency self-check before enabling retries on a task:")
print("  Does running this task twice in a row on the same input change the result?")
print("    MERGE keyed on a real primary key           -> safe to retry")
print("    Blind INSERT / append with no key            -> NOT safe to retry")
print("    A MERGE whose source can contain duplicate keys -> NOT safe until de-duplicated")

---
## Section 8 — Metric Thresholds, In Detail

**INSTRUCTOR — live on screen:** under the Job's **Notifications** settings, alongside the event-type notifications from Section 6, is a **Health** section — this is where metric thresholds live.

### What this actually is

A metric threshold is a rule of the shape *"if `<metric>` `<crosses>` `<value>`, treat that as noteworthy"* — evaluated automatically on every run, without you writing any monitoring code yourself. Two metrics are available today:

| Metric | Measures | Useful for |
|---|---|---|
| `RUN_DURATION_SECONDS` | How long the run took, end to end | Catching a job that's silently degrading — same query, same code, but taking longer and longer each run because a source table has grown, a partition strategy stopped working, or a cluster is undersized |
| `STREAMING_BACKLOG_BYTES` / related backlog metrics | How far behind a streaming task is from the latest available data | Catching an Autoloader/streaming-table task that's fallen behind its source and is quietly not keeping up |

A threshold rule doesn't stop the job — it **triggers whichever notification you attached to "duration warning" / "streaming backlog exceeded"** in Section 6. The two features work together: Section 6 decided *where* the alert goes, this section decides *when* it fires.

### Why this matters more than it looks like it does

Without a metric threshold, a job that goes from "finishes in 10 minutes" to "finishes in 55 minutes" still shows up as a plain **green success** in the Jobs UI — nothing visibly wrong, because nothing *failed*. The team finds out only when someone happens to notice, or when it eventually crosses a hard timeout and actually fails. A duration threshold turns a silent, slow-motion degradation into an active alert the first time it happens — while the job is still succeeding, which is exactly when you have the most time to investigate calmly instead of firefighting an outage.

### A concrete GlobalMart threshold plan

```
Bronze Autoloader task      → RUN_DURATION_SECONDS > 300   (normally ~2 min; 5 min = something's wrong)
Silver SCD2 merge tasks     → RUN_DURATION_SECONDS > 600   (normally ~5-8 min)
Gold fact_sales refresh     → RUN_DURATION_SECONDS > 900   (normally ~10 min; this feeds live dashboards)
```

Set the threshold from real observed run times, not a guess — look at the last 10-20 runs' actual durations first, then set the threshold at roughly 2-3x the normal ceiling. Too tight and you get alert fatigue from normal variance; too loose and it never fires before something's already badly wrong.

In [ ]:
health_rules_example = {
    "rules": [
        {"metric": "RUN_DURATION_SECONDS", "op": "GREATER_THAN", "value": 300},   # Bronze Autoloader task
    ]
}
print(json.dumps(health_rules_example, indent=2))
print()
print("This rule alone changes nothing by itself -- it only fires the")
print("'duration warning' notification configured in Section 6. The threshold")
print("decides WHEN; the notification destination decides WHERE it goes.")

---
## Section 9 — Putting It Together: the Full Live-Demo Script

**INSTRUCTOR — recap the exact click-path, end to end, once, at normal speed, with no new explanation — this is the "watch me do it clean" pass after all the section-by-section detail above:**

1. **Jobs & Pipelines → Jobs → Create Job**
2. Task name: `orchestration` · Type: `Notebook` · Source: `Workspace` · Path: browse to the target notebook
3. Compute → **Add new cluster** → name it → **deselect** Photon → node type **Standard_DS3_v2**
4. **Libraries** tab → add anything the notebook actually imports beyond the runtime defaults (today: none needed)
5. **Parameters** tab → add `catalog_name` = your real catalog. Do **not** add anything named `key`, `secret`, `password`, or `token` here — that's what Secret Scopes are for
6. **Notifications** → on failure, always. On duration warning, for anything feeding a dashboard or downstream job
7. **Retry policy** → 2 retries, 2-minute interval — *after* confirming the task is actually idempotent
8. **Health / metric thresholds** → a duration threshold set from real observed run times, once you have a few runs to look at
9. **Run now** — watch it execute live, show the Run Details page (task graph, cluster logs, duration)

## What's Next

You'll be given a real GlobalMart notebook and asked to orchestrate it yourself, applying every decision from this walkthrough — not just clicking through defaults. Before you start: re-read Day 10 ILT 2's DAG concepts (fan-out/fan-in, no cycles) if you're orchestrating more than one task, since dependencies (`depends_on`) are the one piece of the Jobs UI this session didn't cover in depth — that's Day 10 ILT 3's job, not this one.